# **4. Preprocessing & Feature Engineering**

**Inputs from `2_data_cleaning_fairlending.ipynb`:**
- `hmda_2024_clean.parquet` (8,264,982 × 66) — deduped, outlier-cleaned, **pre-imputation** (used for the sentinel audit + to recover protected/age columns)
- `hmda_2024_cleaned_modelling.parquet` (8,264,982 × 36) — leakage-free feature subset, numeric fields median-imputed, `*_missing` flags where missingness ≥30%, categoricals filled `'Missing'`, `conforming_loan_limit` dropped
- `1_column_dictionary.csv` — the `use_in_model` map driving feature selection

**Focus:**
- **Step 1 — Sentinel / exempt-code audit** on the pre-imputation frame (do the medians in the modeling file hide sentinel codes like `1111`/`8888`/`9999`?).
- **Step 2 — Encode categoricals** (17 low-cardinality fields → one-hot; `1111`→`Exempt`, `8888`→`Unknown`; confirm no geo ID leaked in).
- **Step 3 — Engineer derived features** (`loan_to_income_ratio` with an income≤0 guard; `loan_to_value_ratio` already present; `rate_spread_bucket` computed-but-excluded as leakage; `applicant_age` promoted to a model feature).
- **Step 4 — Train/validation/test split** (72/8/20, stratified on `approved`, moved up from Day 5 so encoders are fit train-only) + `test_demographics_lookup.parquet` (and `val_demographics_lookup.parquet`).
- **Step 5 — Sanity checks** + **Step 6 — Day-4 wrap-up** (with the deferred-EDA scope carried forward).

**Resequencing note:** the full EDA (README Day 3) is deliberately deferred to *after* modeling, not skipped. Only the narrow checks feature engineering needs (cardinality, sentinel sanity) are done here.


In [1]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')

DATA_DIR = '../data/processed/'
MODELLING_DIR = DATA_DIR + 'modelling/'
os.makedirs(MODELLING_DIR, exist_ok=True)

# Dictionary + both source files (row-aligned, same 8,264,982 order)
dict_df = pd.read_csv(DATA_DIR + '1_column_dictionary.csv')
model_df = pd.read_parquet(MODELLING_DIR + 'hmda_2024_cleaned_modelling.parquet')  # 36-col imputed base
clean_df = pd.read_parquet(DATA_DIR + 'hmda_2024_clean.parquet')                   # 66-col pre-imp (age + protected)

assert len(model_df) == len(clean_df), "row misalignment between source files"

# Feature groups from the dictionary
BASE_NUMERIC = dict_df.loc[dict_df['role'] == 'model_numeric', 'column'].tolist()
# applicant_age is promoted to a model categorical in Step 2 (not part of the original
# Day-1 BASE_CAT), so exclude it here to keep this cell idempotent across re-runs.
BASE_CAT    = dict_df.loc[(dict_df['role'] == 'model_categorical') & (dict_df['column'] != 'applicant_age'), 'column'].tolist()
FLAGS       = dict_df.loc[dict_df['role'] == 'model_derived_flag', 'column'].tolist()
TARGET = 'approved'

print(f"model base : {model_df.shape[0]:,} x {model_df.shape[1]}")
print(f"clean      : {clean_df.shape[0]:,} x {clean_df.shape[1]}")
print(f"numeric={len(BASE_NUMERIC)} categorical={len(BASE_CAT)} flags={len(FLAGS)}")


model base : 8,264,982 x 36
clean      : 8,264,982 x 66
numeric=16 categorical=16 flags=3


---
### **Step 1 — Sentinel / Exempt Code Audit**

HMDA's LAR schema uses sentinel codes (`1111` = Exempt, `8888` = Not Applicable, `9999` = No Co-Applicant) that are *not* `NaN` in the raw data, so Day-2's median imputation would not have touched them. If any slipped through as literal numbers, the medians already baked into `hmda_2024_cleaned_modelling.parquet` could be contaminated.

We audit on the **pre-imputation** `hmda_2024_clean.parquet` (the modeling file is already imputed, so it can't show us the raw sentinels). For each numeric suspect we scan for spikes at round sentinel-like numbers; for categoricals we scan for the literal `'1111'`/`'8888'`. If contamination is found we would re-null + re-impute; if not, a clean "checked, none found" note is itself the deliverable.


In [2]:
# Audit on the PRE-imputation clean file (sentinel codes are literal numbers, not NaN)
NUMERIC_SUSPECTS = ['prepayment_penalty_term', 'intro_rate_period', 'total_units', 'total_points_and_fees']
SENTINELS = [1111, 8888, 9999, 999999999, 111111111]

print("=== Numeric sentinel scan (hmda_2024_clean.parquet) ===")
for c in NUMERIC_SUSPECTS:
    s = clean_df[c].dropna()
    hits = {int(k): int(v) for k, v in s.value_counts().items() if k in SENTINELS}
    print(f"  {c:26s} n_unique={s.nunique():>6} sentinel_hits={hits} min={s.min():.0f} max={s.max():.0f}")

print("\n=== Categorical sentinel scan ('1111' Exempt / '8888' Not-Applicable) ===")
CAT_SUSPECTS = BASE_CAT + ['applicant_age']
cat_hits = {}
for c in CAT_SUSPECTS:
    vc = clean_df[c].astype('object').value_counts(dropna=False)
    h = {str(k): int(v) for k, v in vc.items() if str(k) in ('1111', '8888', '9999')}
    if h:
        cat_hits[c] = h
        print(f"  {c:30s} -> {h}")
print("\nCategorical sentinel summary:", cat_hits if cat_hits else "none")


=== Numeric sentinel scan (hmda_2024_clean.parquet) ===
  prepayment_penalty_term    n_unique=    31 sentinel_hits={} min=0 max=360
  intro_rate_period          n_unique=   154 sentinel_hits={} min=1 max=120100
  total_units                n_unique=     4 sentinel_hits={} min=1 max=4
  total_points_and_fees      n_unique= 42261 sentinel_hits={} min=0 max=741049

=== Categorical sentinel scan ('1111' Exempt / '8888' Not-Applicable) ===
  applicant_credit_score_type    -> {'1111': 238185}
  co_applicant_credit_score_type -> {'1111': 238185}
  submission_of_application      -> {'1111': 238032}
  reverse_mortgage               -> {'1111': 238623}
  open_end_line_of_credit        -> {'1111': 237510}
  business_or_commercial_purpose -> {'1111': 237891}
  negative_amortization          -> {'1111': 239039}
  interest_only_payment          -> {'1111': 239039}
  balloon_payment                -> {'1111': 239039}
  applicant_age                  -> {'8888': 213968}

Categorical sentinel summary: 

**Deliverable — sentinel-code audit table** (`column | sentinel checked | contamination | action`)

- `prepayment_penalty_term`, `intro_rate_period`, `total_units` (numeric model cols): **no** `1111/8888/9999` spikes. `intro_rate_period` has a single stray `120100` outlier — non-standard, does **not** move the median, noted only.
- `total_points_and_fees`: not in the model set (`leakage_excluded`), so no model impact even if present.
- 9 model categoricals (`applicant_credit_score_type`, `co_applicant_credit_score_type`, `submission_of_application`, `reverse_mortgage`, `open_end_line_of_credit`, `business_or_commercial_purpose`, `negative_amortization`, `interest_only_payment`, `balloon_payment`): literal `'1111'` = **Exempt** present → a *legitimate category*, not imputation contamination. Relabelled to `'Exempt'` in Step 2.
- `applicant_age`: `'8888'` = Not-Applicable present → relabelled to `'Unknown'` in Step 2.

**Result:** no numeric-sentinel contamination → the Day-2 medians are clean → **no re-imputation required**; the Day-2 modeling file stays valid.


---
### **Step 2 — Encode Categorical Features**

All 17 model categoricals are **low-cardinality** (max 16 levels), so we **one-hot encode** them (no frequency/target encoding needed — no high-cardinality field remains). We fit the encoder **train-only** in Step 4 to stay leakage-safe.

- The literal `'1111'` ("Exempt") sentinel in 9 categoricals is relabelled to a readable `'Exempt'` category before encoding.
- `applicant_age` (HMDA-coded 8-level bucket: `<25, 25-34, 35-44, 45-54, 55-64, 65-74, >74`, plus `8888`) is relabelled `8888`→`'Unknown'` and promoted from `fair_lending_audit_only` to a model categorical.
- **No raw high-cardinality geo ID** is present in the model set — the only `tract_*` columns are numeric *aggregates* (population, % minority, median income, etc.), and `census_tract`/`county_code`/`state_code` were excluded at the Day-1 leakage audit.


In [3]:
# Relabel sentinels to readable categories (symmetric transform, safe pre-split)
EXEMPT, UNKNOWN = '1111', '8888'

for c in BASE_CAT:
    model_df[c] = model_df[c].astype('object').replace(EXEMPT, 'Exempt')

# applicant_age: promote to model categorical, relabel 8888 -> Unknown
model_df['applicant_age'] = clean_df['applicant_age'].astype('object').replace(UNKNOWN, 'Unknown')
CAT_FEATS = BASE_CAT + ['applicant_age']
print(f"Categorical features for encoding ({len(CAT_FEATS)}): {CAT_FEATS}")

# Encoding map (deliverable)
enc_rows = []
for c in CAT_FEATS:
    n = int(model_df[c].nunique(dropna=False))
    note = 'low-card; 1111->Exempt' if c in BASE_CAT else 'promoted from fair_lending_audit_only; 8888->Unknown'
    enc_rows.append({'column': c, 'method': 'one_hot', 'cardinality': n, 'notes': note})
enc_map = pd.DataFrame(enc_rows)
print(enc_map.to_string(index=False))


Categorical features for encoding (17): ['loan_type', 'loan_purpose', 'lien_status', 'occupancy_type', 'construction_method', 'derived_dwelling_category', 'applicant_credit_score_type', 'co_applicant_credit_score_type', 'submission_of_application', 'reverse_mortgage', 'open_end_line_of_credit', 'business_or_commercial_purpose', 'preapproval', 'negative_amortization', 'interest_only_payment', 'balloon_payment', 'applicant_age']
                        column  method  cardinality                                                notes
                     loan_type one_hot            4                               low-card; 1111->Exempt
                  loan_purpose one_hot            6                               low-card; 1111->Exempt
                   lien_status one_hot            2                               low-card; 1111->Exempt
                occupancy_type one_hot            3                               low-card; 1111->Exempt
           construction_method one_hot      

**Deliverable — encoding map** (`column | method | cardinality | notes`) is printed by the cell below; one-line confirmation: **no raw geo ID leaked into the final feature set.**


---
### **Step 3 — Engineer Derived Features**

Per the README's feature-engineering scope, plus the corrections this project's leakage/protected-attribute audit requires:

- `loan_to_income_ratio = loan_amount / income` — **guard first**: Day-2 only nulled *negative* income, leaving **109,336 zero-income rows**. We re-null `income ≤ 0`, re-impute from positive incomes, then divide, and winsorize the ratio at its 99.9th percentile for stability.
- `loan_to_value_ratio` is **already in the model file** (post-cleaning) — the README's "loan-to-property-value ratio" item is already satisfied; no re-derivation.
- `rate_spread_bucket` — binned from `rate_spread` for **EDA only**. `rate_spread` is `leakage_excluded` (only known post-decision), so this bucket is **deliberately kept out of the model matrix** (recorded in the dictionary as `leakage_excluded`).
- `applicant_age_bucket` — `applicant_age` is *already* an HMDA-coded age bucket, so "engineering" it means promoting it to a model feature (after `8888`→`Unknown`). ⚠️ `applicant_age` is a protected attribute under ECOA; it is included here as a standard, legally-permitted credit-risk factor per an explicit project decision, and Day-9 **must** report its disparate impact.


**Note — why `loan_to_value_ratio_missing` is absent from `X_train`.**
The `*_missing` flag rule (`FLAG_THRESHOLD = 0.30`, set in `2_data_cleaning_fairlending.ipynb` Step 5b) is evaluated on the **decision-subset** population the model actually trains on - the same 8,264,982 rows the imputation was computed on - *not* the full 12.2M raw file.

- `loan_to_value_ratio` is **35.25%** missing on the full `hmda_2024_typed.parquet`, but only **10.1% (834,454 rows)** on `hmda_2024_clean.parquet` (the decision subset), because non-decision rows (withdrawn/incomplete) carry far more LTV missingness.
- 10.1% < 30% -> the flag is intentionally **not** created. The column is still median-imputed and fully present (0 nulls) in `X_train`.
- The three flags that *are* present (`debt_to_income_ratio_missing`, `prepayment_penalty_term_missing`, `intro_rate_period_missing`) are >=30% missing in **both** populations, so they survive the rule.

This is by-design, not a silent drop - documented here so the absence of the LTV flag is not mistaken for a bug during the Day-9 fairness review.


In [4]:
# 3.1 loan_to_income_ratio -- guard income<=0 (Day-2 left 109,336 zeros)
income_clean = model_df['income'].mask(model_df['income'] <= 0)
med_income = float(income_clean.median())
income_imp = income_clean.fillna(med_income)
# loan_amount is in DOLLARS; income is in THOUSANDS of dollars -> convert income to dollars
model_df['loan_to_income_ratio'] = model_df['loan_amount'] / (income_imp * 1000)

zeros = int((model_df['income'] <= 0).sum())
lti_cap = float(model_df['loan_to_income_ratio'].quantile(0.999))
model_df['loan_to_income_ratio'] = model_df['loan_to_income_ratio'].clip(upper=lti_cap)
print(f"income<=0 re-nulled: {zeros:,} | re-imputed median income: {med_income:,.0f}")
print(f"loan_to_income_ratio: capped at 99.9th pct = {lti_cap:.2f} | final min={model_df['loan_to_income_ratio'].min():.3f} max={model_df['loan_to_income_ratio'].max():.3f}")

# 3.2 loan_to_value_ratio already present (README LTV item satisfied)
assert 'loan_to_value_ratio' in model_df.columns
print("loan_to_value_ratio already present (README LTV item covered).")

# 3.3 rate_spread_bucket -- COMPUTE for EDA only, EXCLUDED from the model matrix (leakage)
rs = clean_df['rate_spread']
valid_rs = rs[(rs >= 0) & (rs < 9999)]
rate_spread_bucket = pd.cut(valid_rs, bins=[-0.1, 1, 2, 3, 9999],
                            labels=['below_typical', 'typical', 'elevated', 'high']).astype('object')
rate_spread_bucket = rate_spread_bucket.fillna('Unknown')
print("rate_spread_bucket computed (EDA/leakage_excluded ONLY -- not added to model_df):")
print(rate_spread_bucket.value_counts().to_dict())

# 3.4 applicant_age already promoted in Step 2 (no re-binning needed)
print("applicant_age promoted as model categorical (8 levels + Unknown).")

# Update the column dictionary (idempotent: drop prior engineered rows, re-add)
upd = pd.read_csv(DATA_DIR + '1_column_dictionary.csv')
upd = upd[~upd['column'].isin(['loan_to_income_ratio', 'rate_spread_bucket', 'applicant_age'])].copy()
new_rows = pd.DataFrame([
    {'column': 'loan_to_income_ratio', 'role': 'model_derived_numeric', 'use_in_model': 'YES'},
    {'column': 'rate_spread_bucket',   'role': 'leakage_excluded',      'use_in_model': 'NO'},
    {'column': 'applicant_age',        'role': 'model_categorical',     'use_in_model': 'YES'},
])
upd = pd.concat([upd, new_rows], ignore_index=True)
upd.to_csv(DATA_DIR + '1_column_dictionary.csv', index=False)
print(f"\nDictionary updated -> {len(upd)} rows")
print(upd[upd['column'].isin(['loan_to_income_ratio', 'rate_spread_bucket', 'applicant_age'])].to_string(index=False))


income<=0 re-nulled: 109,336 | re-imputed median income: 104
loan_to_income_ratio: capped at 99.9th pct = 42.63 | final min=0.004 max=42.625
loan_to_value_ratio already present (README LTV item covered).
rate_spread_bucket computed (EDA/leakage_excluded ONLY -- not added to model_df):
{'below_typical': 2311759, 'typical': 726744, 'high': 337192, 'elevated': 254881}
applicant_age promoted as model categorical (8 levels + Unknown).

Dictionary updated -> 66 rows
              column                  role use_in_model
loan_to_income_ratio model_derived_numeric          YES
  rate_spread_bucket      leakage_excluded           NO
       applicant_age     model_categorical          YES


---
### **Step 4 — Train / Validation / Test Split (moved up from Day 5)**

Categorical encoding must be fit on a training partition to stay leakage-safe, so the split happens here rather than at the start of Day 5.

- **Stratified on `approved`** (preserves the ~74.6% approval rate across all three partitions).
- **Three partitions, not two:**
  - `X_train` / `y_train` — **72%** — used to *fit* models.
  - `X_val` / `y_val` — **8%** — used **only** for hyperparameter tuning (Optuna), early-stopping, and model selection.
  - `X_test` / `y_test` — **20%** — the final hold-out, used **once** at the very end for reporting metrics + Day-9 fairness. Never touched during tuning (this prevents leaking test information into model selection).
- The 20% test split uses the **same seed and call as before**, so the existing `X_test` rows (and `test_demographics_lookup.parquet`) are preserved exactly.
- One-hot encoder fit on `X_train` only, then applied to all three splits (`handle_unknown='ignore'`).
- `derived_race` / `derived_ethnicity` / `derived_sex` (and `applicant_age`) are **kept out of the feature matrix** and saved to `test_demographics_lookup.parquet` (and `val_demographics_lookup.parquet`), row-index-aligned to `X_test` / `X_val`, so fairness metrics can be joined later without touching model inputs.


In [5]:
# Build the feature matrix (definitions that must precede the split)
NUM_FEATS = BASE_NUMERIC + ['loan_to_income_ratio']
X_cat = model_df[CAT_FEATS].astype('object')
y = model_df[TARGET].astype('int8')
model_df = model_df.reset_index(drop=True)
row_id = np.arange(len(model_df))

from sklearn.model_selection import train_test_split

# --- 3-way stratified split: 72% train / 8% val / 20% test ---
# Step 1: hold out the FINAL test (20%). Same call + seed as before, so the
#         existing X_test rows (and test_demographics_lookup) are preserved exactly.
X_tr0, X_te, y_tr0, y_te, rid_tr0, rid_te = train_test_split(
    X_cat, y, row_id, test_size=0.20, random_state=42, stratify=y)

# Step 2: carve the validation set out of the 80% train (90/10 -> 8% of total).
#         Used ONLY for hyperparameter tuning / Optuna / early-stopping / model selection.
X_tr, X_val, y_tr, y_val, rid_tr, rid_val = train_test_split(
    X_tr0, y_tr0, rid_tr0, test_size=0.10, random_state=42, stratify=y_tr0)

# One-hot encoder fit on the (72%) train only -> apply to train / val / test
ohe = OneHotEncoder(handle_unknown='ignore', dtype=np.float32)
ohe.fit(X_tr)
cat_tr  = pd.DataFrame(ohe.transform(X_tr).toarray(),
                       columns=ohe.get_feature_names_out(CAT_FEATS), index=X_tr.index).reset_index(drop=True)
cat_val = pd.DataFrame(ohe.transform(X_val).toarray(),
                       columns=ohe.get_feature_names_out(CAT_FEATS), index=X_val.index).reset_index(drop=True)
cat_te  = pd.DataFrame(ohe.transform(X_te).toarray(),
                       columns=ohe.get_feature_names_out(CAT_FEATS), index=X_te.index).reset_index(drop=True)

num_tr  = model_df.loc[X_tr.index,  NUM_FEATS + FLAGS].reset_index(drop=True).astype(np.float32)
num_val = model_df.loc[X_val.index, NUM_FEATS + FLAGS].reset_index(drop=True).astype(np.float32)
num_te  = model_df.loc[X_te.index,  NUM_FEATS + FLAGS].reset_index(drop=True).astype(np.float32)

X_train = pd.concat([num_tr,  cat_tr ], axis=1).astype(np.float32)
X_val   = pd.concat([num_val, cat_val], axis=1).astype(np.float32)
X_test  = pd.concat([num_te,  cat_te ], axis=1).astype(np.float32)

print(f"X_train: {X_train.shape[0]:,} x {X_train.shape[1]} | X_val: {X_val.shape[0]:,} x {X_val.shape[1]} | X_test: {X_test.shape[0]:,} x {X_test.shape[1]}")
print(f"y_train approved% = {y_tr.mean():.4f} | y_val approved% = {y_val.mean():.4f} | y_test approved% = {y_te.mean():.4f}")

X_train.to_parquet(MODELLING_DIR + 'X_train.parquet', index=False)
X_val.to_parquet(MODELLING_DIR + 'X_val.parquet', index=False)
X_test.to_parquet(MODELLING_DIR + 'X_test.parquet', index=False)
y_tr.to_frame('approved').to_parquet(MODELLING_DIR + 'y_train.parquet', index=False)
y_val.to_frame('approved').to_parquet(MODELLING_DIR + 'y_val.parquet', index=False)
y_te.to_frame('approved').to_parquet(MODELLING_DIR + 'y_test.parquet', index=False)

# Demographics lookup (protected + age), row-index-aligned to X_test (official fairness set)
demo = clean_df.loc[X_te.index, ['derived_race', 'derived_ethnicity', 'derived_sex', 'applicant_age']].copy()
demo = demo.reset_index(drop=True)
demo['row_id'] = rid_te
demo.to_parquet(MODELLING_DIR + 'test_demographics_lookup.parquet', index=False)

# Same lookup for the val partition (optional early fairness sanity; test stays official)
demo_val = clean_df.loc[X_val.index, ['derived_race', 'derived_ethnicity', 'derived_sex', 'applicant_age']].copy()
demo_val = demo_val.reset_index(drop=True)
demo_val['row_id'] = rid_val
demo_val.to_parquet(MODELLING_DIR + 'val_demographics_lookup.parquet', index=False)

print("\nSaved: X_train, X_val, X_test, y_train, y_val, y_test, test_demographics_lookup, val_demographics_lookup")
print("Test lookup:", demo.shape, "cols:", list(demo.columns))
print("Val  lookup:", demo_val.shape, "cols:", list(demo_val.columns))


X_train: 5,950,786 x 103 | X_val: 661,199 x 103 | X_test: 1,652,997 x 103
y_train approved% = 0.7465 | y_val approved% = 0.7465 | y_test approved% = 0.7465

Saved: X_train, X_val, X_test, y_train, y_val, y_test, test_demographics_lookup, val_demographics_lookup
Test lookup: (1652997, 5) cols: ['derived_race', 'derived_ethnicity', 'derived_sex', 'applicant_age', 'row_id']
Val  lookup: (661199, 5) cols: ['derived_race', 'derived_ethnicity', 'derived_sex', 'applicant_age', 'row_id']


---
### **Step 5 — Final Preprocessing Sanity Checks**

- `X_train`, `X_val`, and `X_test` have identical columns, dtypes, and encoding (the classic silent bug when encoding is applied per-split).
- Re-confirm no outcome-derived / post-decision column (e.g. `rate_spread_bucket`) and no protected `derived_*` column is present in the feature matrix.
- Report final feature count, dtype breakdown, and train/val/test row counts + test demographic group sizes.


In [6]:
# 5.1 schema match across all three partitions
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), "Column mismatch across partitions!"
# 5.2 no leakage / protected cols in any partition matrix
PROTECTED = ['derived_race', 'derived_ethnicity', 'derived_sex']
for part_name, part in [('train', X_train), ('val', X_val), ('test', X_test)]:
    assert not any(c in PROTECTED for c in part.columns), f"Protected column leaked into {part_name} matrix!"
    assert 'rate_spread_bucket' not in part.columns, f"Leakage column in {part_name} matrix!"
# 5.3 report
print("Feature count :", X_train.shape[1])
print("Dtypes        :", X_train.dtypes.value_counts().to_dict())
print("Train rows    :", f"{len(X_train):,}", "| Val rows:", f"{len(X_val):,}", "| Test rows:", f"{len(X_test):,}")
print("Train approved%:", round(float(y_tr.mean()), 4),
      "| Val approved%:", round(float(y_val.mean()), 4),
      "| Test approved%:", round(float(y_te.mean()), 4))
print("\nTest demographic group sizes (derived_race):", demo['derived_race'].value_counts().to_dict())
print("Test demographic group sizes (derived_sex):", demo['derived_sex'].value_counts().to_dict())
print("\nSanity checks PASSED.")


Feature count : 103
Dtypes        : {dtype('float32'): 103}
Train rows    : 5,950,786 | Val rows: 661,199 | Test rows: 1,652,997
Train approved%: 0.7465 | Val approved%: 0.7465 | Test approved%: 0.7465

Test demographic group sizes (derived_race): {'White': 1062536, 'Race Not Available': 287999, 'Black or African American': 145119, 'Asian': 100127, 'Joint': 36657, 'American Indian or Alaska Native': 12068, '2 or more minority races': 4085, 'Native Hawaiian or Other Pacific Islander': 3990, 'Free Form Text Only': 416}
Test demographic group sizes (derived_sex): {'Joint': 577372, 'Male': 557987, 'Female': 372203, 'Sex Not Available': 145435}

Sanity checks PASSED.


---
### **Step 6 — Wrap-up**

**Day-4 summary**
- Engineered features added: `loan_to_income_ratio` (numeric, model), `applicant_age` (categorical, model — promoted), `rate_spread_bucket` (EDA-only, excluded).
- Encoding: 17 low-cardinality categoricals one-hot (fit train-only); no frequency/target encoding required; no geo ID in the feature set.
- Train/validation/test split: 72/8/20 stratified on `approved`; `X_train` / `X_val` / `X_test` / `y_train` / `y_val` / `y_test` / `test_demographics_lookup.parquet` / `val_demographics_lookup.parquet` saved.
- Fairness-relevant columns (`derived_race`, `derived_ethnicity`, `derived_sex`, `applicant_age`) are preserved out-of-band for Day 9.

**Deferred-EDA scope (carried forward, not dropped):** class-balance check on the *actual* split, distribution checks on the newly engineered features, and demographic group-size confirmation ahead of Day 9.

**End-of-Day-4 Checklist**
- [x] Sentinel/exempt-code audit completed for `MODEL_COLUMNS`, contamination ruled out (numeric); categorical `1111`/`8888` relabelled
- [x] Categorical features encoded (one-hot for low-cardinality; none high-cardinality remained)
- [x] Confirmed no raw high-cardinality geo ID present in the final feature set
- [x] `loan_to_income_ratio`, `applicant_age` engineered and added to the column dictionary (`rate_spread_bucket` added as `leakage_excluded`)
- [x] Train/validation/test split created (72/8/20), stratified on `approved`, with demographic group sizes checked in the test split
- [x] One-hot encoder fit train-only and applied to all three splits (train/val/test)
- [x] `test_demographics_lookup.parquet` saved, row-index-aligned to `X_test`
- [x] `val_demographics_lookup.parquet` saved, row-index-aligned to `X_val`
- [x] `X_train`/`X_val`/`X_test` schema-matched (same columns, dtypes, encoding)
- [x] Day-4 summary + deferred-EDA scope note written

**Not for Day 4 (save for later days):** model training, hyperparameter tuning, SHAP/explainability, fairness metric computation (Day 9), and the full formal EDA report (deliberately deferred, not skipped).
